# MyKepatuhan — RAG Evaluation (DeepEval)

**Goal:** Measure retrieval quality, answer faithfulness, and latency across English and Bahasa Melayu compliance questions.

**Metrics (DeepEval equivalents):**
- `FaithfulnessMetric` — is the answer grounded in retrieved documents (no hallucination)?
- `AnswerRelevancyMetric` — does the answer actually address the question?
- `ContextualPrecisionMetric` — are the retrieved chunks actually relevant?
- `ContextualRecallMetric` — did we find all the relevant chunks?
- Latency — how long does each pipeline stage take? (unchanged)

**Documents ingested (~180 pages):**
- Perlepasan cukai perniagaan (business tax exemptions)
- Business registration guides
- SSM official documents

## Cell 1 — Install Dependencies

In [1]:
# DeepEval replaces RAGAS entirely — no datasets library needed
%pip install deepeval --quiet
%pip install llama-index-llms-ollama llama-index-embeddings-ollama --quiet

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Cell 2 — Imports & Config

In [16]:
import os, time, json, warnings
import pandas as pd
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv()

# ── DeepEval imports (replaces ragas imports) ────────────────────────
from deepeval import evaluate
from deepeval.metrics import (
    FaithfulnessMetric,       # replaces Faithfulness
    AnswerRelevancyMetric,    # replaces AnswerRelevancy
    ContextualPrecisionMetric, # replaces ContextPrecision
    ContextualRecallMetric,   # replaces ContextRecall
)
from deepeval.test_case import LLMTestCase
# DeepEvalBaseLLM not needed — using GeminiModel directly


# ── DeepEval needs an LLM judge — wrap your Ollama model ────────────
# DeepEval uses its own LLM interface, so we create a thin wrapper
# around the same gemma4:e4b model you already use.

# Gemini as judge — unbiased since your RAG uses a different model (gemma4:e4b)
# Add GEMINI_API_KEY to your .env file
# Get one free at: https://aistudio.google.com/apikey
from deepeval.models import GeminiModel

judge_llm = GeminiModel(
    model="gemini-2.5-flash-lite",      
    api_key=os.getenv("GEMINI_KEY"),
)

# ── Metrics (same four as RAGAS, threshold=0.5 is DeepEval default) ──
# async_mode=False tells DeepEval to skip its internal asyncio.timeout() wrapper
# and run each metric synchronously — required when using a local Ollama judge
# that can't respond within DeepEval's hardcoded async deadline.
METRICS = [
    FaithfulnessMetric(threshold=0.5, model=judge_llm, include_reason=True, async_mode=False),
    AnswerRelevancyMetric(threshold=0.5, model=judge_llm, include_reason=True, async_mode=False),
    ContextualPrecisionMetric(threshold=0.5, model=judge_llm, include_reason=True, async_mode=False),
    ContextualRecallMetric(threshold=0.5, model=judge_llm, include_reason=True, async_mode=False),
]

METRIC_NAMES = ["faithfulness", "answer_relevancy", "contextual_precision", "contextual_recall"]

print("✅ DeepEval metrics initialised ")

✅ DeepEval metrics initialised 


## Cell 3 — Load Your Existing Retriever

Point this to your `retriever.py` — no changes needed to the pipeline.

In [17]:
import sys
# Adjust this path to wherever your retriever.py lives
sys.path.append("../backend/pipeline/")

from backend.pipeline.retriever import build_query_engine, build_retriever

# Build one general engine (no metadata filters) for evaluation
query_engine = build_query_engine()
retriever    = build_retriever()

print("✅ Query engine and retriever loaded")

✅ Query engine and retriever loaded


## Cell 4 — Test Dataset

20 questions spanning English and Bahasa Melayu compliance topics. Unchanged from RAGAS version.

In [18]:
TEST_DATASET = [
    # ── ENGLISH — MyIPO ────────────────────────────────────
    {
        "question": "What is the primary legal status of the Intellectual Property Corporation of Malaysia?",
        "ground_truth": "According to Section 3 of the Act, the Corporation is established as a body corporate with perpetual succession and a common seal, which may sue and be sued in its corporate name.",
        "topic": "registration",
        "authority": "MyIPO",
        "lang": "en",
    },
    {
        "question": "Which intellectual property laws are administered by the Corporation under the First Schedule?",
        "ground_truth": "The First Schedule lists the Trade Marks Act 1976, Patents Act 1983, Copyright Act 1987, Industrial Designs Act 1996, Layout-Designs of Integrated Circuits Act 2000, and the Geographical Indications Act 2000.",
        "topic": "registration",
        "authority": "MyIPO",
        "lang": "en",
    },
    {
        "question": "Who is responsible for appointing the Director General of the Corporation?",
        "ground_truth": "The Minister is responsible for appointing the Director General of the Intellectual Property Corporation of Malaysia, determining their terms and conditions of service.",
        "topic": "employment",
        "authority": "MyIPO",
        "lang": "en",
    },
    {
        "question": "How long is the term of office for an appointed member of the Corporation?",
        "ground_truth": "An appointed member of the Corporation holds office for a term not exceeding three years, as specified in their instrument of appointment, and is eligible for reappointment.",
        "topic": "employment",
        "authority": "MyIPO",
        "lang": "en",
    },
    {
        "question": "How can a member of the Corporation resign from their position?",
        "ground_truth": "According to Section 8, a member may resign from their office at any time by giving a written notice addressed directly to the Minister.",
        "topic": "employment",
        "authority": "MyIPO",
        "lang": "en",
    },
    {
        "question": "Are the Controller and Deputy Controllers of Copyright part of the Corporation's officers?",
        "ground_truth": "Yes, under the Second Schedule of the Act, the Controller, Deputy Controllers, and Assistant Controllers of Copyright are officially listed as officers of the Corporation.",
        "topic": "registration",
        "authority": "MyIPO",
        "lang": "en",
    },
    {
        "question": "Did the Trademarks Act 2019 affect the Intellectual Property Corporation of Malaysia Act?",
        "ground_truth": "Yes, a note in the document indicates that references to the older Trade Marks Act 1976 are subject to the Trademarks Act 2019 (Act 815), which came into operation on 27 December 2019.",
        "topic": "registration",
        "authority": "MyIPO",
        "lang": "en",
    },
    {
        "question": "Does the Corporation have its own financial fund?",
        "ground_truth": "Yes, the Act establishes a specific fund to be administered and controlled by the Corporation to finance its ongoing operations, pay remunerations, and cover administrative expenses.",
        "topic": "tax",
        "authority": "MyIPO",
        "lang": "en",
    },
    {
        "question": "Who sits on the membership board of the Corporation?",
        "ground_truth": "The Corporation's membership consists of a Chairman, the Director General, and other appointed members selected for their expertise in intellectual property, law, or business.",
        "topic": "registration",
        "authority": "MyIPO",
        "lang": "en",
    },
    {
        "question": "What is the official Act number for the Intellectual Property Corporation of Malaysia Act 2002?",
        "ground_truth": "The official Act number is Act 617. It received Royal Assent on 14 January 2002 and was published in the Gazette on 24 January 2002.",
        "topic": "registration",
        "authority": "MyIPO",
        "lang": "en",
    },
    # ── BAHASA MELAYU — SSM ───────────────────────────────
    {
        "question": "Berapakah fi pendaftaran untuk mendaftarkan perniagaan milikan tunggal yang menggunakan nama sendiri seperti di kad pengenalan?",
        "ground_truth": "Fi pendaftaran bagi perniagaan milikan tunggal yang menggunakan nama sendiri seperti di kad pengenalan adalah sebanyak RM 30 setahun.",
        "topic": "registration",
        "authority": "SSM",
        "lang": "bm",
    },
    {
        "question": "Apakah syarat umur minimum untuk mendaftarkan perniagaan baru di Malaysia?",
        "ground_truth": "Pemilik atau rakan kongsi perniagaan mestilah seorang warganegara atau penduduk tetap yang berumur lapan belas (18) tahun dan ke atas.",
        "topic": "registration",
        "authority": "SSM",
        "lang": "bm",
    },
    {
        "question": "Apakah jenis-jenis perubahan maklumat perniagaan yang perlu didaftarkan kepada SSM?",
        "ground_truth": "Perubahan maklumat perniagaan yang boleh didaftarkan termasuklah perubahan alamat perniagaan, perubahan jenis perniagaan, perubahan maklumat cawangan, dan perubahan maklumat pemilikan (seperti kemasukan, tarik diri, atau kematian rakan kongsi).",
        "topic": "registration",
        "authority": "SSM",
        "lang": "bm",
    },
    {
        "question": "Apakah denda jika pemilik meneruskan perniagaan tanpa mendaftarkan perubahan maklumat perniagaannya?",
        "ground_truth": "Di bawah Akta Pendaftaran Perniagaan 1956, kegagalan mendaftarkan perubahan adalah satu kesalahan yang boleh didenda tidak melebihi RM 10,000 atau dipenjara tidak melebihi satu (1) tahun, atau kedua-duanya sekali jika disabitkan kesalahan.",
        "topic": "compliance",
        "authority": "SSM",
        "lang": "bm",
    },
    {
        "question": "Siapakah yang layak untuk memohon Skim Pendaftaran Perniagaan Percuma (SPPP) di bawah kategori Usahawan Kumpulan B40?",
        "ground_truth": "Golongan yang layak ialah warganegara berumur 18 tahun ke atas, merupakan penerima bantuan khas kerajaan seperti BPR, BKM, atau mySalam, dan tiada rekod entiti perniagaan yang masih aktif di SSM.",
        "topic": "registration",
        "authority": "SSM",
        "lang": "bm",
    },

]

print(f"✅ Test dataset ready: {len(TEST_DATASET)} questions")
print(f"   English: {sum(1 for q in TEST_DATASET if q['lang'] == 'en')}")
print(f"   Bahasa Melayu: {sum(1 for q in TEST_DATASET if q['lang'] == 'bm')}")

topics = {}
for q in TEST_DATASET:
    topics[q['topic']] = topics.get(q['topic'], 0) + 1
print(f"   Topics: {topics}")

✅ Test dataset ready: 15 questions
   English: 10
   Bahasa Melayu: 5
   Topics: {'registration': 10, 'employment': 3, 'tax': 1, 'compliance': 1}


## Cell 5 — Latency Benchmark (Per Component)

Unchanged — latency measurement is independent of the eval framework.

In [19]:
import nest_asyncio
nest_asyncio.apply()

BENCHMARK_QUESTION = "How long is the term of office for an appointed member of the Corporation?"
latency_results = {}

print(f"Benchmarking with: '{BENCHMARK_QUESTION}'\n")

# ── 1. Retrieval latency ─────────────────────────────────────────────
t0 = time.perf_counter()
nodes = retriever.retrieve(BENCHMARK_QUESTION)
retrieval_time = time.perf_counter() - t0
latency_results["retrieval_ms"] = round(retrieval_time * 1000, 1)
print(f"  Retrieval:   {latency_results['retrieval_ms']} ms  ({len(nodes)} nodes returned)")

# ── 2. End-to-end query latency (retrieval + rerank + generation) ────
t0 = time.perf_counter()
response = query_engine.query(BENCHMARK_QUESTION)
e2e_time = time.perf_counter() - t0
latency_results["end_to_end_ms"] = round(e2e_time * 1000, 1)
latency_results["llm_generation_ms"] = round((e2e_time - retrieval_time) * 1000, 1)

print(f"  LLM generation + rerank: {latency_results['llm_generation_ms']} ms")
print(f"  End-to-end TOTAL: {latency_results['end_to_end_ms']} ms")
print(f"\nSample answer:\n{response.response[:300]}...")

Benchmarking with: 'How long is the term of office for an appointed member of the Corporation?'

  Retrieval:   4441.3 ms  (15 nodes returned)
  LLM generation + rerank: 11387.6 ms
  End-to-end TOTAL: 15828.9 ms

Sample answer:
An appointed member of the Corporation holds office for a term not exceeding three years, subject to any conditions specified in their instrument of appointment and provided they do not resign, vacate the office, or have their appointment revoked sooner. They may be eligible for reappointment....


## Cell 6 — Full Latency Across All Questions

Unchanged — latency measurement is independent of the eval framework.

In [20]:
latency_records = []

print(f"Running {len(TEST_DATASET)} questions...\n")

for i, item in enumerate(TEST_DATASET):
    t0 = time.perf_counter()
    try:
        resp = query_engine.query(item["question"])
        elapsed_ms = (time.perf_counter() - t0) * 1000
        latency_records.append({
            "question": item["question"][:60] + "...",
            "lang": item["lang"],
            "topic": item["topic"],
            "latency_ms": round(elapsed_ms, 1),
            "answer_len": len(resp.response),
        })
        print(f"  [{i+1:02d}] {item['lang'].upper()} | {item['topic']:<12} | {elapsed_ms:.0f} ms")
    except Exception as e:
        print(f"  [{i+1:02d}] ERROR: {e}")

df_latency = pd.DataFrame(latency_records)

print("\n── Latency Summary ─────────────────────────────")
print(f"  Mean:    {df_latency['latency_ms'].mean():.0f} ms")
print(f"  Median:  {df_latency['latency_ms'].median():.0f} ms")
print(f"  P95:     {df_latency['latency_ms'].quantile(0.95):.0f} ms")
print(f"  Max:     {df_latency['latency_ms'].max():.0f} ms")
print()
print("── By Language ─────────────────────────────────")
print(df_latency.groupby('lang')['latency_ms'].agg(['mean','median','max']).round(0))

Running 15 questions...

  [01] EN | registration | 13577 ms
  [02] EN | registration | 2313 ms
  [03] EN | employment   | 1979 ms
  [04] EN | employment   | 2429 ms
  [05] EN | employment   | 6608 ms
  [06] EN | registration | 30564 ms
  [07] EN | registration | 17231 ms
  [08] EN | tax          | 11842 ms
  [09] EN | registration | 13967 ms
  [10] EN | registration | 3079 ms
  [11] BM | registration | 11005 ms
  [12] BM | registration | 3563 ms
  [13] BM | registration | 3135 ms
  [14] BM | compliance   | 4614 ms
  [15] BM | registration | 2412 ms

── Latency Summary ─────────────────────────────
  Mean:    8555 ms
  Median:  4614 ms
  P95:     21231 ms
  Max:     30564 ms

── By Language ─────────────────────────────────
         mean  median      max
lang                          
bm     4946.0  3563.0  11005.0
en    10359.0  9225.0  30564.0


## Cell 7 — Collect RAG Outputs & Build DeepEval Test Cases

RAGAS used a HuggingFace `Dataset`. DeepEval uses `LLMTestCase` objects instead.
The fields map directly:

| RAGAS field    | DeepEval field         |
|----------------|------------------------|
| `question`     | `input`                |
| `answer`       | `actual_output`        |
| `contexts`     | `retrieval_context`    |
| `ground_truth` | `expected_output`      |

In [21]:
test_cases  = []  
rows_meta   = []  
errors      = []

print(f"Collecting RAG outputs for DeepEval evaluation...\n")

for i, item in enumerate(TEST_DATASET):
    try:
        response = query_engine.query(item["question"])

        contexts = [
            node.node.text
            for node in response.source_nodes
            if node.node.text.strip()
        ]

        if not contexts:
            print(f"  [{i+1:02d}] ⚠️  No contexts retrieved — skipping")
            errors.append(item["question"])
            continue

        # ── Build a DeepEval LLMTestCase ─────────────────────────────
        test_case = LLMTestCase(
            input=item["question"],           # replaces: "question"
            actual_output=response.response,  # replaces: "answer"
            retrieval_context=contexts,        # replaces: "contexts"
            expected_output=item["ground_truth"],  # replaces: "ground_truth"
        )
        test_cases.append(test_case)
        rows_meta.append({"lang": item["lang"], "topic": item["topic"]})

        print(f"  [{i+1:02d}] ✅  {item['lang'].upper()} | {item['topic']:<12} | {len(contexts)} contexts")

    except Exception as e:
        print(f"  [{i+1:02d}] ❌  ERROR: {e}")
        errors.append(item["question"])

print(f"\n✅ Built {len(test_cases)}/{len(TEST_DATASET)} test cases")
if errors:
    print(f"❌ Failed: {len(errors)} questions")


  [01] ✅  EN | registration | 3 contexts
  [02] ✅  EN | registration | 3 contexts
  [03] ✅  EN | employment   | 3 contexts
  [04] ✅  EN | employment   | 3 contexts
  [05] ✅  EN | employment   | 3 contexts
  [06] ✅  EN | registration | 3 contexts
  [07] ✅  EN | registration | 3 contexts
  [08] ✅  EN | tax          | 3 contexts
  [09] ✅  EN | registration | 3 contexts
  [10] ✅  EN | registration | 3 contexts
  [11] ✅  BM | registration | 3 contexts
  [12] ✅  BM | registration | 3 contexts
  [13] ✅  BM | registration | 3 contexts
  [14] ✅  BM | compliance   | 3 contexts
  [15] ✅  BM | registration | 3 contexts

✅ Built 15/15 test cases


## Cell 8 — Run DeepEval Evaluation

Replaces `ragas.evaluate()` with `deepeval.evaluate()`.
The API is almost identical — pass test cases + metrics.

In [23]:
print("Running DeepEval evaluation — calling .measure() directly, no async...\n")

# We call metric.measure(test_case) directly instead of deepeval.evaluate().
# This bypasses ALL of DeepEval's internal asyncio.timeout() wrappers,
# which were cancelling our slow local Ollama judge before it could respond.

metric_map = {
    "FaithfulnessMetric":          "faithfulness",
    "AnswerRelevancyMetric":        "answer_relevancy",
    "ContextualPrecisionMetric":    "contextual_precision",
    "ContextualRecallMetric":       "contextual_recall",
}

records = []

for i, (tc, meta) in enumerate(zip(test_cases, rows_meta)):
    print(f"  [{i+1:02d}/{len(test_cases)}] {meta['lang'].upper()} | {meta['topic']}")
    row = {
        "question": tc.input,
        "lang":     meta["lang"],
        "topic":    meta["topic"],
    }
    for metric in METRICS:
        col = metric_map.get(type(metric).__name__, type(metric).__name__)
        try:
            metric.measure(tc)          # plain synchronous call
            row[col] = metric.score
            print(f"       {col:<26} {metric.score:.3f}")
        except Exception as e:
            print(f"       {col:<26} ERROR: {e}")
            row[col] = None
    records.append(row)

df_scores = __import__('pandas').DataFrame(records)
SCORE_COLS = ["faithfulness", "answer_relevancy", "contextual_precision", "contextual_recall"]

print("\n✅ Evaluation complete")


Output()

Running DeepEval evaluation — calling .measure() directly, no async...

  [01/15] EN | registration


Output()

       faithfulness               1.000


Output()

...

Output()

       answer_relevancy           ERROR: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 4.936691105s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global

Output()

       contextual_precision       ERROR: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 4.756801434s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global

       contextual_recall          ERROR: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 4.553540331s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global

## Cell 9 — Detailed Results Breakdown

In [10]:
# ── Build a results DataFrame from DeepEval output ───────────────────
# DeepEval stores per-metric scores on each test_case after evaluate()

metric_map = {
    "FaithfulnessMetric":         "faithfulness",
    "AnswerRelevancyMetric":       "answer_relevancy",
    "ContextualPrecisionMetric":   "contextual_precision",
    "ContextualRecallMetric":      "contextual_recall",
}

records = []
for tc, meta in zip(test_cases, rows_meta):
    row = {"question": tc.input, "lang": meta["lang"], "topic": meta["topic"]}
    for metric in METRICS:
        col = metric_map.get(type(metric).__name__, type(metric).__name__)
        row[col] = metric.score  # DeepEval stores the last-run score on the metric object
    records.append(row)

df_scores = pd.DataFrame(records)
SCORE_COLS = ["faithfulness", "answer_relevancy", "contextual_precision", "contextual_recall"]

# ── Overall scores ───────────────────────────────────────────────────
print("═" * 55)
print("OVERALL SCORES")
print("═" * 55)
for m in SCORE_COLS:
    if m in df_scores.columns:
        score = df_scores[m].mean()
        bar   = "█" * int(score * 20)
        print(f"  {m:<26} {score:.3f}  {bar}")

# ── By language ──────────────────────────────────────────────────────
print("\n" + "═" * 55)
print("SCORES BY LANGUAGE")
print("═" * 55)
lang_scores = df_scores.groupby("lang")[SCORE_COLS].mean().round(3)
print(lang_scores.to_string())

# ── By topic ─────────────────────────────────────────────────────────
print("\n" + "═" * 55)
print("SCORES BY TOPIC")
print("═" * 55)
topic_scores = df_scores.groupby("topic")[SCORE_COLS].mean().round(3)
print(topic_scores.to_string())

# ── Worst-performing questions ───────────────────────────────────────
print("\n" + "═" * 55)
print("LOWEST FAITHFULNESS QUESTIONS (potential hallucination)")
print("═" * 55)
if "faithfulness" in df_scores.columns:
    worst = df_scores.nsmallest(5, "faithfulness")[["question", "faithfulness", "lang", "topic"]]
    for _, row in worst.iterrows():
        print(f"  [{row['faithfulness']:.2f}] {row['question'][:70]}")

# ── DeepEval bonus: failure reasons ─────────────────────────────────
# DeepEval surfaces per-case reasons when include_reason=True
print("\n" + "═" * 55)
print("FAILURE REASONS (DeepEval-only feature)")
print("═" * 55)
for tc in test_cases:
    failed = [m for m in METRICS if not m.is_successful()]
    if failed:
        print(f"\nQ: {tc.input[:70]}")
        for m in failed:
            print(f"  ✗ {type(m).__name__}: {m.reason}")

═══════════════════════════════════════════════════════
OVERALL SCORES
═══════════════════════════════════════════════════════
  faithfulness               1.000  ████████████████████
  answer_relevancy           1.000  ████████████████████
  contextual_precision       1.000  ████████████████████
  contextual_recall          1.000  ████████████████████

═══════════════════════════════════════════════════════
SCORES BY LANGUAGE
═══════════════════════════════════════════════════════
      faithfulness  answer_relevancy  contextual_precision  contextual_recall
lang                                                                         
bm             1.0               1.0                   1.0                1.0
en             1.0               1.0                   1.0                1.0

═══════════════════════════════════════════════════════
SCORES BY TOPIC
═══════════════════════════════════════════════════════
              faithfulness  answer_relevancy  contextual_precision  cont

## Cell 10 — Bilingual Comparison

In [11]:
print("═" * 55)
print("BILINGUAL PERFORMANCE COMPARISON")
print("═" * 55)

en_scores = df_scores[df_scores["lang"] == "en"][SCORE_COLS].mean()
bm_scores = df_scores[df_scores["lang"] == "bm"][SCORE_COLS].mean()

comparison = pd.DataFrame({
    "English": en_scores,
    "Bahasa Melayu": bm_scores,
    "Gap (EN - BM)": en_scores - bm_scores,
}).round(3)

print(comparison.to_string())

if (en_scores - bm_scores).abs().mean() > 0.1:
    print("\n⚠️  Significant bilingual gap detected.")
    print("   Consider: multilingual embeddings or BM-specific chunking")
else:
    print("\n✅ Bilingual performance is consistent (gap < 0.1)")

═══════════════════════════════════════════════════════
BILINGUAL PERFORMANCE COMPARISON
═══════════════════════════════════════════════════════
                      English  Bahasa Melayu  Gap (EN - BM)
faithfulness              1.0            1.0            0.0
answer_relevancy          1.0            1.0            0.0
contextual_precision      1.0            1.0            0.0
contextual_recall         1.0            1.0            0.0

✅ Bilingual performance is consistent (gap < 0.1)


## Cell 11 — Save Results

In [12]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

summary = {
    "timestamp":   timestamp,
    "n_questions": len(df_scores),
    "overall": {m: round(df_scores[m].mean(), 4) for m in SCORE_COLS if m in df_scores.columns},
    "by_language": lang_scores.to_dict(),
    "by_topic":    topic_scores.to_dict(),
    "latency": {
        "mean_ms":   round(df_latency["latency_ms"].mean(), 1),
        "median_ms": round(df_latency["latency_ms"].median(), 1),
        "p95_ms":    round(df_latency["latency_ms"].quantile(0.95), 1),
    }
}

with open(f"eval_results_{timestamp}.json", "w") as f:
    json.dump(summary, f, indent=2)

df_scores.to_csv(f"eval_scores_{timestamp}.csv", index=False)
df_latency.to_csv(f"eval_latency_{timestamp}.csv", index=False)

print(f"✅ Saved: eval_results_{timestamp}.json")
print(f"✅ Saved: eval_scores_{timestamp}.csv")
print(f"✅ Saved: eval_latency_{timestamp}.csv")

✅ Saved: eval_results_20260524_0037.json
✅ Saved: eval_scores_20260524_0037.csv
✅ Saved: eval_latency_20260524_0037.csv


## Cell 12 — LinkedIn Metrics Generator

In [13]:
overall       = summary["overall"]
latency       = summary["latency"]
n_docs        = 180
n_authorities = 4

faith     = overall.get('faithfulness', 0)
relevancy = overall.get('answer_relevancy', 0)
precision = overall.get('contextual_precision', 0)  # renamed from context_precision
recall    = overall.get('contextual_recall', 0)     # renamed from context_recall
p95       = latency['p95_ms']
median    = latency['median_ms']

print("═" * 65)
print("LINKEDIN-READY BULLET POINTS")
print("═" * 65)
print(f"""
📌 Resume / LinkedIn Project Description:

  Built MyKepatuhan — an end-to-end RAG-powered compliance assistant
  for Malaysian entrepreneurs navigating SSM, LHDN, and municipal
  regulations. Ingested {n_docs}+ pages of official government documents
  across {n_authorities} authorities into a hybrid BM25+dense vector
  index (Pinecone). Achieved {faith:.0%} faithfulness and
  {relevancy:.0%} answer relevancy (DeepEval) with bilingual EN/BM
  support, delivering cited compliance answers at {median:.0f}ms
  median / {p95:.0f}ms P95 latency.

📌 Key Metrics to Screenshot:
  - Faithfulness:           {faith:.3f}  (target > 0.80)
  - Answer Relevancy:       {relevancy:.3f}  (target > 0.75)
  - Contextual Precision:   {precision:.3f}  (target > 0.70)
  - Contextual Recall:      {recall:.3f}  (target > 0.70)
  - Median Query Latency:   {median:.0f} ms
  - P95 Query Latency:      {p95:.0f} ms
  - Documents Indexed:      {n_docs}+ pages across {n_authorities} authorities
  - Bilingual: English + Bahasa Melayu
""")

print("═" * 65)
print("WHAT TO DO NEXT BASED ON YOUR SCORES:")
print("═" * 65)

if faith < 0.75:
    print("  ⚠️  Faithfulness < 0.75 → LLM is hallucinating.")
    print("      Fix: Add 'Answer ONLY based on the provided context.'")
    print("      to your system prompt in retriever.py")
if precision < 0.70:
    print("  ⚠️  Contextual Precision < 0.70 → Retrieval is noisy.")
    print("      Fix: Lower similarity_top_k from 15 → 8, increase HYBRID_ALPHA")
if recall < 0.65:
    print("  ⚠️  Contextual Recall < 0.65 → Missing relevant chunks.")
    print("      Fix: Increase similarity_top_k, check chunk sizes")
if relevancy < 0.75:
    print("  ⚠️  Answer Relevancy < 0.75 → Answers are off-topic.")
    print("      Fix: Improve the query engine prompt to stay on topic")
if faith >= 0.80 and relevancy >= 0.75 and precision >= 0.70 and recall >= 0.70:
    print("  ✅  All metrics look solid. Time to ingest the full document set!")

═════════════════════════════════════════════════════════════════
LINKEDIN-READY BULLET POINTS
═════════════════════════════════════════════════════════════════

📌 Resume / LinkedIn Project Description:

  Built MyKepatuhan — an end-to-end RAG-powered compliance assistant
  for Malaysian entrepreneurs navigating SSM, LHDN, and municipal
  regulations. Ingested 180+ pages of official government documents
  across 4 authorities into a hybrid BM25+dense vector
  index (Pinecone). Achieved 100% faithfulness and
  100% answer relevancy (DeepEval) with bilingual EN/BM
  support, delivering cited compliance answers at 2657ms
  median / 14159ms P95 latency.

📌 Key Metrics to Screenshot:
  - Faithfulness:           1.000  (target > 0.80)
  - Answer Relevancy:       1.000  (target > 0.75)
  - Contextual Precision:   1.000  (target > 0.70)
  - Contextual Recall:      1.000  (target > 0.70)
  - Median Query Latency:   2657 ms
  - P95 Query Latency:      14159 ms
  - Documents Indexed:      180+ pa